# Doubly Linked List

Each node holds a value plus `prev` and `next` pointers. The extra link costs
memory but buys backward traversal and O(1) deletion when you already hold a
reference to the node - no need to walk the list to find its predecessor.

| Operation | Singly | Doubly |
|---|---|---|
| Insert / delete at front | O(1) | O(1) |
| Delete a node you hold a reference to | O(n) - find predecessor first | O(1) |
| Traverse backwards | not possible | O(n) |
| Pointers per node | 1 | 2 |

In [ ]:
class ListNode:
     def __init__(self, val=0):
         self.val = val
         self.prev, self.next = None, None

def to_list(head):
    ls = []
    curr = head.next
    while curr:
        ls.append(curr.val)
        curr = curr.next
    return ls

We'll use a dummy head for all functions.

### Insert at front

Four links to fix instead of the two a singly linked list needs: the new node's `next` and
`prev`, the dummy's `next`, and the `prev` of the node that *was* first - unless the list
was empty, which is what the `if new.next` guard covers.

Order matters just as it does in a singly linked list: read `head.next` into the new node
before overwriting it.

**Time:** O(1) &nbsp; **Space:** O(1)

In [ ]:
def insert_front(head, val):
    new = ListNode(val)
    new.next = head.next
    head.next = new
    new.prev = head
    if new.next:  # empty list has no old first node to repair
        new.next.prev = new


def test_insert_front():
    head = ListNode(-1)
    insert_front(head, 1)
    insert_front(head, 2)
    insert_front(head, 3)
    assert to_list(head) == [3, 2, 1]
    # prev links point back: 3 -> dummy, 2 -> 3, 1 -> 2
    assert head.next.prev is head
    assert head.next.next.prev is head.next


test_insert_front()

### Insert at end

Still O(n). The `prev` pointers do not help here - there is no tail pointer, so the
end has to be found by walking. Starting the walk from the dummy rather than
`head.next` lets the empty list use the same code, since the dummy is then the last
node.

A production doubly linked list keeps a tail pointer (or a second sentinel) and makes
this O(1). `collections.deque` is exactly that, which is why the built-in section
below matters more than this function.

**Time:** O(n) &nbsp; **Space:** O(1)

In [ ]:
def insert_end(head, val):
    new = ListNode(val)
    curr = head
    while curr.next:  # walk to the last node; the dummy is it when list is empty
        curr = curr.next
    curr.next = new
    new.prev = curr


def test_insert_end():
    head = ListNode(-1)
    insert_end(head, 1)  # into an empty list
    assert to_list(head) == [1]
    insert_end(head, 2)
    insert_end(head, 3)
    assert to_list(head) == [1, 2, 3]
    # prev links chain back to the dummy
    assert head.next.prev is head
    assert head.next.next.prev is head.next


test_insert_end()

### Delete front

Hop the dummy over the first node, then repair the new first node's `prev` so it
points back at the dummy. The inner `if head.next` handles deleting the only node,
where there is no new first node to repair.

Forgetting that inner repair is the classic doubly linked list bug: forward traversal
still works, so tests that only walk `next` never notice.

**Time:** O(1) &nbsp; **Space:** O(1)

In [ ]:
def delete_front(head):
    if head.next:
        head.next = head.next.next
        if head.next:  # list may now be empty
            head.next.prev = head


def test_delete_front():
    head = ListNode(-1)
    insert_end(head, 1)
    insert_end(head, 2)
    insert_end(head, 3)
    delete_front(head)
    assert to_list(head) == [2, 3]
    assert head.next.prev is head

    # delete down to empty, then once more
    delete_front(head)
    delete_front(head)
    assert to_list(head) == []
    delete_front(head)
    assert head.next is None


test_delete_front()

### Delete end

With a tail pointer this would be the one-liner `tail = tail.prev` - the whole reason
to pay for `prev` pointers. Without one we are back to walking, stopping one node
short via `curr.next.next`.

The empty-list guard has to come first, otherwise `curr.next.next` dereferences
`None`.

**Time:** O(n) &nbsp; **Space:** O(1)

In [ ]:
def delete_end(head):
    if head.next is None:  # empty list
        return
    curr = head
    while curr.next.next:  # stop on the second-last node
        curr = curr.next
    curr.next = None


def test_delete_end():
    head = ListNode(-1)
    insert_end(head, 1)
    insert_end(head, 2)
    insert_end(head, 3)
    delete_end(head)
    assert to_list(head) == [1, 2]

    # down to a single element, then empty, then a no-op
    delete_end(head)
    assert to_list(head) == [1]
    delete_end(head)
    assert to_list(head) == []
    delete_end(head)
    assert head.next is None


test_delete_end()

### Reverse

Easier than reversing a singly linked list: every node already holds both pointers,
so reversal is just **swapping them** - no three-pointer dance.

The subtlety is how to keep walking. After the swap, the node you wanted to visit
next is no longer in `curr.next` (that now holds the old `prev`); it is in
`curr.prev`.

```
1 ⇄ 2 ⇄ 3        swap prev/next on each node

node 1:  prev=None next=2     →   prev=2     next=None
node 2:  prev=1    next=3     →   prev=3     next=1
node 3:  prev=2    next=None  →   prev=None  next=2

prev ends on node 3 - the old tail, which is the new head
head.next = 3,  3.prev = head

result:  D ⇄ 3 ⇄ 2 ⇄ 1
```

The first node is detached from the dummy before the loop (`curr.prev = None`).
Without that, the swap leaves the new tail's `next` pointing back at the dummy, the
list becomes a cycle, and `to_list` never terminates.

**Time:** O(n) &nbsp; **Space:** O(1)

In [ ]:
def reverse(head):
    curr = head.next
    if curr:
        curr.prev = None  # detach from dummy so the new tail terminates the list
    prev = None
    while curr:
        # remember curr: when the loop ends, prev holds the old last node,
        # which becomes the new first node
        prev = curr
        # reversing a DLL node just means swapping its two pointers
        curr.prev, curr.next = curr.next, curr.prev
        # after the swap the old next is reachable through prev
        curr = curr.prev
    head.next = prev
    if prev:
        prev.prev = head


def test_reverse():
    head = ListNode(-1)
    insert_end(head, 1)
    insert_end(head, 2)
    insert_end(head, 3)
    reverse(head)
    assert to_list(head) == [3, 2, 1]
    # prev links are consistent in the reversed list
    assert head.next.prev is head
    assert head.next.next.prev is head.next

    # reversing twice restores the original order
    reverse(head)
    assert to_list(head) == [1, 2, 3]

    # single element and empty list
    head = ListNode(-1)
    insert_end(head, 1)
    reverse(head)
    assert to_list(head) == [1]

    head = ListNode(-1)
    reverse(head)
    assert to_list(head) == []


test_reverse()

# Python Built-in: `collections.deque`

Python's `collections.deque` is implemented as a **doubly-linked list** of fixed-size blocks.

| Operation | deque | list |
|-----------|-------|------|
| Append right | O(1) | O(1) amortized |
| Append left | O(1) | O(n) - shifts all elements |
| Pop right | O(1) | O(1) |
| Pop left | O(1) | O(n) - shifts all elements |

Use `deque` whenever you need efficient operations on both ends.

In [ ]:
from collections import deque

d = deque([1, 2, 3])

d.appendleft(0)   # O(1) - [0, 1, 2, 3]
d.append(4)        # O(1) - [0, 1, 2, 3, 4]
d.popleft()        # O(1) - returns 0, deque is [1, 2, 3, 4]
d.pop()            # O(1) - returns 4, deque is [1, 2, 3]

# rotate: move n elements from one end to the other
d.rotate(1)        # [3, 1, 2] - right rotate
d.rotate(-1)       # [1, 2, 3] - left rotate

# bounded deque: automatically discards from opposite end
bounded = deque(maxlen=3)
for i in range(5):
    bounded.append(i)
print(list(bounded))  # [2, 3, 4] - oldest elements dropped